# 02 – Feature Engineering

| Campo | Detalle |
|---|---|
| **Input** | `data_hotel_booking/hotel_booking_clean.csv` |
| **Output** | `data_hotel_booking/train_fe.csv`, `data_hotel_booking/val_fe.csv`, `data_hotel_booking/test_fe.csv` |
| **Split temporal** | Train = julio 2015 → junio 2017 · Val = julio 2017 · Test = agosto 2017 |


### ¿Por qué este esquema de split?

El split es **temporal y estricto**: Train siempre precede a Val, y Val siempre precede a Test.
Esto replica exactamente cómo funciona el modelo en producción: el hotel solo puede
usar información del pasado para predecir el futuro.

| Set | Período | Rol |
|---|---|---|
| **Train** | Hasta mes n-2 | Aprender patrones y ajustar estadísticos |
| **Val** | Mes n-1 | Elegir el mejor modelo y el umbral óptimo |
| **Test** | Mes n | Evaluación final — se ejecuta **una sola vez** |

Un split aleatorio sería incorrecto aquí porque mezclaría reservas de agosto 2017
con reservas de 2015, permitiendo que el modelo "vea el futuro" durante el
entrenamiento e inflando artificialmente las métricas.


In [ ]:
# Instalar de ser necesario
# %pip install pandas numpy scikit-learn scipy

In [21]:
# ── Imports ───
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import linregress
import warnings
warnings.filterwarnings('ignore')

# ── Paths ───
data_path = Path.cwd() / "data_hotel_booking"
data_path.mkdir(parents=True, exist_ok=True)


# 1. Carga y preparación de split temporal

In [22]:
df = pd.read_csv(data_path / "hotel_booking_clean.csv")
print(f"Shape de entrada: {df.shape}")
print(df.dtypes)


Shape de entrada: (118969, 31)
hotel                                 str
is_canceled                           str
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                            int64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
assigned_room_type                    str
booking_changes                     int64
deposit_type                          str
age

In [ ]:
# ── Mapeo de meses a número (soporta inglés y español) ──────────────────────
MONTH_MAP = {
    # Inglés 
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12,
    # Español 
    'Enero': 1, 'Febrero': 2, 'Marzo': 3, 'Abril': 4,
    'Mayo': 5, 'Junio': 6, 'Julio': 7, 'Agosto': 8,
    'Septiembre': 9, 'Octubre': 10, 'Noviembre': 11, 'Diciembre': 12,
}

df['arrival_date_month_num'] = df['arrival_date_month'].map(MONTH_MAP)

df['arrival_date'] = pd.to_datetime(
    df[['arrival_date_year', 'arrival_date_month_num', 'arrival_date_day_of_month']]
    .rename(columns={'arrival_date_year': 'year',
                     'arrival_date_month_num': 'month',
                     'arrival_date_day_of_month': 'day'})
)
df = df.sort_values('arrival_date').reset_index(drop=True)
# ── Target numérico ──────────────────────────────────────────────────────────
cancel_map = {'No cancelado': 0, 'Cancelado': 1, 'no cancelado': 0, 'cancelado': 1}

if df['is_canceled'].astype(str).str.contains('cancelado', case=False, na=False).any():
    df['is_canceled_num'] = df['is_canceled'].map(cancel_map)
else:
    df['is_canceled_num'] = df['is_canceled'].astype(int)

print(f"is_canceled_num check: {df['is_canceled_num'].value_counts().to_dict()}")

is_canceled_num check: {0: 74963, 1: 44006}


# 2. Split temporal (sin data leakage)

In [24]:
# ── Identificar los dos últimos periodos ────────────────────────────────────
periodos = (df[['arrival_date_year', 'arrival_date_month_num']]
            .drop_duplicates()
            .sort_values(['arrival_date_year', 'arrival_date_month_num'])
            .reset_index(drop=True))

ult_mes_anio, ult_mes   = periodos.iloc[-1]   # Test  (mes n)
ant_mes_anio, ant_mes   = periodos.iloc[-2]   # Val   (mes n-1)

cond_test = (df['arrival_date_year'] == ult_mes_anio) & (df['arrival_date_month_num'] == ult_mes)
cond_val  = (df['arrival_date_year'] == ant_mes_anio) & (df['arrival_date_month_num'] == ant_mes)

df_test  = df[cond_test].copy()
df_val   = df[cond_val].copy()
df_train = df[~(cond_test | cond_val)].copy()

print(f"Rango Train: {df_train['arrival_date'].min().date()} → {df_train['arrival_date'].max().date()} | {len(df_train):,} filas")
print(f"Rango Val  : {df_val['arrival_date'].min().date()}   → {df_val['arrival_date'].max().date()}   | {len(df_val):,} filas")
print(f"Rango Test : {df_test['arrival_date'].min().date()}   → {df_test['arrival_date'].max().date()}   | {len(df_test):,} filas")


Rango Train: 2015-07-01 → 2017-06-30 | 108,738 filas
Rango Val  : 2017-07-01   → 2017-07-31   | 5,308 filas
Rango Test : 2017-08-01   → 2017-08-31   | 4,923 filas


# 3. Feature Engineering


Esta sección construye todas las variables nuevas que entrarán al modelo.
Se divide en dos bloques con lógicas distintas, ambas diseñadas para respetar
el orden temporal y evitar data leakage.



### 3.1 Variables de ventana móvil y lag

Estas variables capturan la **dinámica temporal** del dataset: inercia de
cancelaciones, ritmo de entrada de reservas, tendencia de precios.

Se calculan sobre el dataset completo **ordenado cronológicamente**, usando
`.shift(1)` para garantizar que cada reserva solo "vea" su propio pasado —
nunca el presente ni el futuro.

Una vez calculadas, se re-divide el dataset en Train/Val/Test usando los mismos
índices del split temporal — los valores ya están calculados correctamente.


### 3.2 Estadísticos aprendidos solo en Train

Estas variables requieren aprender un parámetro del dato (una media, un
percentil, una tasa) antes de transformar. El proceso tiene dos pasos
explícitos y separados:

**Paso A — Aprendizaje (solo Train):**
Se calculan todos los estadísticos de referencia usando únicamente `df_train`.
Val y Test no existen en este momento.

**Paso B — Aplicación (función `apply_fe`):**
Una única función transforma cualquier split usando los estadísticos
congelados del Paso A. No re-aprende nada. Se llama tres veces:
una para Train, una para Val, una para Test.

### 3.3 Variables de ventana móvil y lag (calculadas sobre dataset completo ordenado)


In [ ]:
# ┌─────────────────────────────────────────────────────────────────────────┐
# │  ESTAS VARIABLES SE CALCULAN SOBRE df_full ORDENADO CRONOLÓGICAMENTE   │
# └─────────────────────────────────────────────────────────────────────────┘

# 1. rolling_cancel_7d: inercia de cancelaciones de los últimos 7 días
def compute_rolling_cancel_7d(data):
    return (data['is_canceled_num']
            .rolling(window=7, min_periods=1)
            .mean()
            .shift(1))


# 2. lead_time_lag_market
def compute_lead_time_lag_market(data):
    """
    lead_time_lag_market: promedio de anticipación de las últimas 10 reservas
    dentro del mismo segmento de mercado
    
    Calculado SIN data leakage: solo ve el pasado de ESTE split
    """
    return (data.groupby('market_segment')['lead_time']
            .transform(lambda x: x.shift(1)
                       .rolling(window=10, min_periods=1)
                       .mean()))


# 3. Demmand acceleration
def compute_demand_acceleration(data):
    """
    demand_acceleration: ratio MA3 / MA7 de reservas por día de llegada
    Mide si la demanda se está acelerando o frenando
    
    Calculado SIN data leakage: resample solo sobre ESTE split
    """
    res_count = (data.set_index('arrival_date')
                 .resample('D')
                 .size()
                 .to_frame('res_count'))
    res_count['ma3'] = res_count['res_count'].rolling(3, min_periods=1).mean()
    res_count['ma7'] = res_count['res_count'].rolling(7, min_periods=1).mean()
    res_count['demand_acceleration'] = res_count['ma3'] / (res_count['ma7'] + 1)
    
    return data.merge(res_count[['demand_acceleration']],
                      left_on='arrival_date', right_index=True, how='left')


# 4. Price_trend_slope
def compute_price_trend_slope(data):
    """
    price_trend_slope: pendiente del precio en los últimos 15 días (por hotel)
    
    Calculado SIN data leakage: solo ve el pasado de ESTE split por hotel
    """
    def get_slope(x):
        if len(x) < 5:
            return 0.0
        try:
            slope, *_ = linregress(range(len(x)), x)
            return slope
        except:
            return 0.0
    
    return data.assign(
        price_trend_slope=data.groupby('hotel')['adr']
                             .transform(lambda x: x.shift(1)
                                       .rolling(window=15, min_periods=5)
                                       .apply(get_slope, raw=True))
    )


#Aplico als funciones
print("Aplicando rolling cancel 7d...")
df_train['rolling_cancel_7d'] = compute_rolling_cancel_7d(df_train)
df_val['rolling_cancel_7d']   = compute_rolling_cancel_7d(df_val)
df_test['rolling_cancel_7d']  = compute_rolling_cancel_7d(df_test)


print("Aplicando lead_time_lag_market...")
df_train['lead_time_lag_market'] = compute_lead_time_lag_market(df_train)
df_val['lead_time_lag_market']   = compute_lead_time_lag_market(df_val)
df_test['lead_time_lag_market']  = compute_lead_time_lag_market(df_test)


print("\nAplicando demand_acceleration...")
df_train = compute_demand_acceleration(df_train)
df_val   = compute_demand_acceleration(df_val)
df_test  = compute_demand_acceleration(df_test)


print("\nAplicando price_trend_slope...")
df_train = compute_price_trend_slope(df_train)
df_val   = compute_price_trend_slope(df_val)
df_test  = compute_price_trend_slope(df_test)

print("✅ Variables temporales calculadas SIN data leakage")

Aplicando rolling cancel 7d...
Aplicando lead_time_lag_market...

Aplicando demand_acceleration...

Aplicando price_trend_slope...
✅ Variables temporales calculadas SIN data leakage


### 3.4 Estadísticos aprendidos SOLO en Train (luego aplicados a Val/Test)

In [26]:
# ══════════════════════════════════════════════════════════════════════════════
# PASO A: APRENDIZAJE — TODO SE AJUSTA ÚNICAMENTE CON df_train
# ══════════════════════════════════════════════════════════════════════════════

# A1. ADR promedio y std por (hotel, mes) → para adr_nom_diff
adr_stats = (df_train
             .groupby(['hotel', 'arrival_date_month_num'])['adr']
             .agg(mean_adr='mean', std_adr='std')
             .reset_index())

# A2. Percentil 95 de lead_time por segmento de mercado → riesgo de cancelación
p95_lead = (df_train
            .groupby('market_segment')['lead_time']
            .quantile(0.95)
            .to_dict())

# A3. Mediana de ADR por hotel → posicionamiento de precio relativo
p50_adr = (df_train
           .groupby('hotel')['adr']
           .quantile(0.5)
           .to_dict())

# A4. Tasa histórica de cancelación por (hotel, customer_type) → prior bayesiano
cancel_rate = (df_train
               .groupby(['hotel', 'customer_type'])['is_canceled_num']
               .mean()
               .reset_index()
               .rename(columns={'is_canceled_num': 'cancel_rate_segment'}))

# A5. ADR promedio por (hotel, assigned_room_type) → upgrade penalty
adr_room = (df_train
            .groupby(['hotel', 'assigned_room_type'])['adr']
            .mean()
            .reset_index()
            .rename(columns={'adr': 'adr_room_avg'}))

print("✅ Estadísticos de Train calculados:")
print(f"  adr_stats filas: {len(adr_stats)}")
print(f"  p95_lead keys: {list(p95_lead.keys())[:5]}")
print(f"  p50_adr: {p50_adr}")
print(f"  cancel_rate filas: {len(cancel_rate)}")


✅ Estadísticos de Train calculados:
  adr_stats filas: 24
  p95_lead keys: ['Agencia de viajes offline', 'Agencia de viajes online', 'Aviación', 'Complementario', 'Corporativo']
  p50_adr: {'City Hotel': 97.75, 'Resort Hotel': 70.75}
  cancel_rate filas: 8


In [27]:
# ══════════════════════════════════════════════════════════════════════════════
# PASO B: APLICACIÓN — función que transforma cualquier split usando los
#         estadísticos aprendidos en Train
# ══════════════════════════════════════════════════════════════════════════════

def apply_fe(data: pd.DataFrame) -> pd.DataFrame:
    """Aplica toda la feature engineering a un split usando estadísticos de Train.

    NUNCA re-aprende nada; solo transforma.
    """
    d = data.copy()

    # ── Variables derivadas simples (no dependen de estadísticos externos) ──
    d['total_nights']          = d['stays_in_weekend_nights'] + d['stays_in_week_nights']
    d['expected_revenue_loss'] = d['adr'] * d['total_nights']
    d['net_history_score']     = ((d['previous_bookings_not_canceled'] - d['previous_cancellations']) /
                                  (d['previous_bookings_not_canceled'] + d['previous_cancellations'] + 1))
    d['room_upgrade']          = (d['reserved_room_type'] != d['assigned_room_type']).astype(int)
    d['has_children']          = ((d.get('children', 0) > 0) | (d.get('babies', 0) > 0)).astype(int)
    d['high_demand_season']    = d['arrival_date_month_num'].isin([6, 7, 8, 12]).astype(int)
    d['weekend_ratio']         = (d['stays_in_weekend_nights'] /
                                  (d['total_nights'] + 1))
    d['booking_intensity']     = d['total_of_special_requests'] + d['booking_changes']

    # ── Variables que usan estadísticos de Train ──────────────────────────────

    # adr_nom_diff: cuánto paga esta reserva vs la media del hotel·mes
    d = d.merge(adr_stats, on=['hotel', 'arrival_date_month_num'], how='left')
    d['adr_nom_diff'] = d['adr'] - d['mean_adr']
    d['adr_z_score']  = (d['adr'] - d['mean_adr']) / (d['std_adr'] + 1e-6)

    # lead_time_high_risk: p95 del segmento calculado en Train
    d['lead_time_high_risk'] = d.apply(
        lambda x: 1 if x['lead_time'] > p95_lead.get(x['market_segment'], 1e9) else 0,
        axis=1
    )

    # adr_above_median: mediana por hotel calculada en Train
    d['adr_above_median'] = d.apply(
        lambda x: 1 if x['adr'] > p50_adr.get(x['hotel'], 0) else 0,
        axis=1
    )

    # cancel_rate_segment: prior bayesiano de Train
    d = d.merge(cancel_rate, on=['hotel', 'customer_type'], how='left')
    d['cancel_rate_segment'] = d['cancel_rate_segment'].fillna(
        d['cancel_rate_segment'].median() if d['cancel_rate_segment'].notna().any() else 0.3
    )

    # adr_vs_room_avg: cuánto paga esta reserva vs el promedio de ese tipo de habitación
    d = d.merge(adr_room, on=['hotel', 'assigned_room_type'], how='left')
    d['adr_vs_room_avg'] = d['adr'] - d['adr_room_avg'].fillna(d['adr'])

    return d


# ── Aplicar a los tres splits ─────────────────────────────────────────────────
df_train_fe = apply_fe(df_train)
df_val_fe   = apply_fe(df_val)
df_test_fe  = apply_fe(df_test)

print(f"Train FE shape : {df_train_fe.shape}")
print(f"Val   FE shape : {df_val_fe.shape}")
print(f"Test  FE shape : {df_test_fe.shape}")


Train FE shape : (108738, 55)
Val   FE shape : (5308, 55)
Test  FE shape : (4923, 55)


# 4. Sanity checks anti-leakage

In [28]:
# ── 4.1 Comprobación temporal: Train < Val < Test ────────────────────────────
assert df_train_fe['arrival_date'].max() < df_val_fe['arrival_date'].min(), \
    "❌ LEAKAGE: Train se solapa con Val"
assert df_val_fe['arrival_date'].max() < df_test_fe['arrival_date'].min(), \
    "❌ LEAKAGE: Val se solapa con Test"
print("✅ Orden temporal correcto: Train < Val < Test")

# ── 4.2 Los estadísticos se calcularon solo sobre Train ──────────────────────
print(f"\nVerificación estadísticos de Train:")
print(f"  adr_stats calculado con {len(df_train):,} filas (Train)")
print(f"  df_val contiene {len(df_val):,} filas – NO usadas para aprendizaje")
print(f"  df_test contiene {len(df_test):,} filas – NO usadas para aprendizaje")

# ── 4.3 Nulos e infinitos ────────────────────────────────────────────────────
def check_nulls_inf(name, df_):
    inf_count  = np.isinf(df_.select_dtypes(include=np.number)).sum().sum()
    null_count = df_.isnull().sum().sum()
    print(f"  {name}: {null_count:,} nulos | {inf_count:,} infinitos")

print("\nNulos e infinitos por split:")
check_nulls_inf("Train", df_train_fe)
check_nulls_inf("Val  ", df_val_fe)
check_nulls_inf("Test ", df_test_fe)


✅ Orden temporal correcto: Train < Val < Test

Verificación estadísticos de Train:
  adr_stats calculado con 108,738 filas (Train)
  df_val contiene 5,308 filas – NO usadas para aprendizaje
  df_test contiene 4,923 filas – NO usadas para aprendizaje

Nulos e infinitos por split:
  Train: 39 nulos | 0 infinitos
  Val  : 38 nulos | 0 infinitos
  Test : 38 nulos | 0 infinitos


In [ ]:
# ── 4.4 Lógica de negocio ────────────────────────────────────────────────────
# Verifica que no hayan sobrevivido registros físicamente imposibles
# después de aplicar toda la feature engineering.

for name, d in [('Train', df_train_fe), ('Val', df_val_fe), ('Test', df_test_fe)]:
    ceros = ((d['adults'] + d.get('children', pd.Series(0, index=d.index)) +
              d.get('babies', pd.Series(0, index=d.index))) == 0).sum()
    neg_adr = (d['adr'] < 0).sum()
    neg_lt  = (d['lead_time'] < 0).sum()
    print(f"{name}: {ceros} reservas sin huéspedes | {neg_adr} ADR<0 | {neg_lt} lead_time<0")


Train: 0 reservas sin huéspedes | 1 ADR<0 | 0 lead_time<0
Val: 0 reservas sin huéspedes | 0 ADR<0 | 0 lead_time<0
Test: 0 reservas sin huéspedes | 0 ADR<0 | 0 lead_time<0


# 5. Eliminación de leakage y encoding antes de guardar

Antes de guardar la data resultante para modelar, se realizan dos operaciones finales:

1. **Drop de columnas de leakage** — columnas que no pueden estar disponibles
   al momento de predecir o que son duplicados del target.
2. **LabelEncoding seguro** — las variables categóricas se convierten a entero.
   El encoder se ajusta (`fit`) **únicamente sobre Train** y se aplica en modo
   `transform` a Val y Test. Categorías no vistas en Train reciben el valor `-1`.

Esto garantiza que los CSVs que entran al `03_Modelling` ya están listos para
entrenar sin ningún paso de preparación adicional.


In [30]:
from sklearn.preprocessing import LabelEncoder

# ── Columnas de leakage a eliminar ───────────────────────────────────────────
# Estas columnas NO pueden estar disponibles al momento de predecir:
#   · is_canceled          : texto duplicado del target numérico
#   · reservation_status   : estado final (Check-Out / Canceled) — leakage puro
#   · reservation_status_date : fecha en que se registró la cancelación
#   · arrival_date         : datetime ya representado por sus partes numéricas
COLS_LEAKAGE = [
    'is_canceled',
    'reservation_status',
    'reservation_status_date',
    'arrival_date',
]

def drop_leakage(df_):
    return df_.drop(columns=[c for c in COLS_LEAKAGE if c in df_.columns],
                    errors='ignore')

df_train_fe = drop_leakage(df_train_fe)
df_val_fe   = drop_leakage(df_val_fe)
df_test_fe  = drop_leakage(df_test_fe)

print("✅ Columnas de leakage eliminadas.")
print(f"   Columnas restantes: {df_train_fe.shape[1]}")


✅ Columnas de leakage eliminadas.
   Columnas restantes: 51


In [31]:
# ── LabelEncoding seguro ─────────────────────────────────────────────────────
# fit SOLO en Train → transform en Val y Test
# Categorías no vistas en Train → -1

TARGET = 'is_canceled_num'
encoders = {}

cat_cols = df_train_fe.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != TARGET]

for col in cat_cols:
    le = LabelEncoder()
    df_train_fe[col] = le.fit_transform(df_train_fe[col].astype(str))
    encoders[col] = le

    df_val_fe[col] = df_val_fe[col].astype(str).map(
        lambda s: int(le.transform([s])[0]) if s in le.classes_ else -1
    )
    df_test_fe[col] = df_test_fe[col].astype(str).map(
        lambda s: int(le.transform([s])[0]) if s in le.classes_ else -1
    )

print(f"✅ LabelEncoding aplicado a {len(cat_cols)} columnas categóricas:")
for col in cat_cols:
    print(f"   · {col}")


✅ LabelEncoding aplicado a 10 columnas categóricas:
   · hotel
   · arrival_date_month
   · meal
   · country
   · market_segment
   · distribution_channel
   · reserved_room_type
   · assigned_room_type
   · deposit_type
   · customer_type


# 6. Guardado de los tres CSVs

In [32]:
# ── Eliminar columnas helper intermedias de cálculo ─────────────────────────
cols_drop_fe = ['mean_adr', 'std_adr', 'adr_room_avg']

def clean_and_save(df_, name):
    out = df_.drop(columns=[c for c in cols_drop_fe if c in df_.columns], errors='ignore')
    # Limpiar inf / nan residuales
    num_cols = out.select_dtypes(include='number').columns
    out[num_cols] = out[num_cols].replace([float('inf'), float('-inf')], float('nan')).fillna(0)
    path = data_path / f"{name}_fe.csv"
    out.to_csv(path, index=False)
    print(f"  Guardado: {path}  →  shape {out.shape}")
    return out

print("Guardando splits...")
df_train_fe = clean_and_save(df_train_fe, 'train')
df_val_fe   = clean_and_save(df_val_fe,   'val')
df_test_fe  = clean_and_save(df_test_fe,  'test')

print("\n✅ CSVs listos para modelado.")
print(f"   Columnas guardadas: {[c for c in df_train_fe.columns]}")


Guardando splits...
  Guardado: c:\Users\yair.barnatan\Desktop\Proyecto Hoteles\data_hotel_booking\train_fe.csv  →  shape (108738, 48)
  Guardado: c:\Users\yair.barnatan\Desktop\Proyecto Hoteles\data_hotel_booking\val_fe.csv  →  shape (5308, 48)
  Guardado: c:\Users\yair.barnatan\Desktop\Proyecto Hoteles\data_hotel_booking\test_fe.csv  →  shape (4923, 48)

✅ CSVs listos para modelado.
   Columnas guardadas: ['hotel', 'lead_time', 'arrival_date_year', 'arrival_date_month', 'arrival_date_week_number', 'arrival_date_day_of_month', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'meal', 'country', 'market_segment', 'distribution_channel', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'reserved_room_type', 'assigned_room_type', 'booking_changes', 'deposit_type', 'agent', 'days_in_waiting_list', 'customer_type', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'arrival_date_month_num', 'is_canceled_num', 

# 7. Resumen de features generadas

In [33]:
# ── Resumen final del output de Feature Engineering ──────────────────────────
print("=" * 55)
print("  OUTPUT — 02_FeatEng")
print("=" * 55)

for name, df_ in [('Train', df_train_fe), ('Val', df_val_fe), ('Test = a predecir', df_test_fe)]:
    cols_sin_target = [c for c in df_.columns if c != 'is_canceled_num']
    print(f"\n  {name}")
    print(f"    Filas   : {df_.shape[0]:,}")
    print(f"    Columnas: {len(cols_sin_target)}  (sin contar target)")

print("\n" + "=" * 55)
print("  Columnas finales (sin target):")
print("=" * 55)
for i, col in enumerate(cols_sin_target, 1):
    print(f"  {i:02d}. {col}")

  OUTPUT — 02_FeatEng

  Train
    Filas   : 108,738
    Columnas: 47  (sin contar target)

  Val
    Filas   : 5,308
    Columnas: 47  (sin contar target)

  Test = a predecir
    Filas   : 4,923
    Columnas: 47  (sin contar target)

  Columnas finales (sin target):
  01. hotel
  02. lead_time
  03. arrival_date_year
  04. arrival_date_month
  05. arrival_date_week_number
  06. arrival_date_day_of_month
  07. stays_in_weekend_nights
  08. stays_in_week_nights
  09. adults
  10. children
  11. babies
  12. meal
  13. country
  14. market_segment
  15. distribution_channel
  16. is_repeated_guest
  17. previous_cancellations
  18. previous_bookings_not_canceled
  19. reserved_room_type
  20. assigned_room_type
  21. booking_changes
  22. deposit_type
  23. agent
  24. days_in_waiting_list
  25. customer_type
  26. adr
  27. required_car_parking_spaces
  28. total_of_special_requests
  29. arrival_date_month_num
  30. rolling_cancel_7d
  31. lead_time_lag_market
  32. demand_acceleratio

In [34]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import textwrap
import os

output_dir  = "Figures/FeatEng"
output_file = "01_Variables_finales_modelo.png"
os.makedirs(output_dir, exist_ok=True)
path_final  = os.path.join(output_dir, output_file)

# ── Datos ─────────────────────────────────────────────────────────────────────
target_row = ["Target", "is_canceled_num  →  1 = Cancelado  |  0 = No cancelado"]

data_raw = [
    ["Demanda y Segmentación",
     "adults, children, babies, country, customer_type, market_segment,\n"
     "distribution_channel  +  has_children, cancel_rate_segment"],

    ["Temporalidad",
     "arrival_date_year, arrival_date_month_num, arrival_date_week_number,\n"
     "arrival_date_day_of_month  +  high_demand_season"],

    ["Estancia",
     "stays_in_weekend_nights, stays_in_week_nights\n"
     "+  total_nights, weekend_ratio"],

    ["Anticipación y Demanda",
     "lead_time  +  lead_time_high_risk, lead_time_lag_market, demand_acceleration"],

    ["Fidelidad y Riesgo",
     "is_repeated_guest, previous_cancellations, previous_bookings_not_canceled\n"
     "+  net_history_score"],

    ["Producto y Operación",
     "hotel, reserved_room_type, meal, required_car_parking_spaces,\n"
     "total_of_special_requests  +  room_upgrade, booking_intensity"],

    ["Financiera",
     "adr  +  adr_nom_diff, adr_z_score, adr_above_median,\n"
     "adr_vs_room_avg, expected_revenue_loss, price_trend_slope"],

    ["Dinámica Temporal (Lags)",
     "rolling_cancel_7d, demand_acceleration, price_trend_slope\n"
     "(calculadas con shift=1 sobre dataset ordenado cronológicamente)"],
]

# ── Canvas ────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 12))
ax.set_xlim(0, 10)
ax.set_ylim(-1, len(data_raw) + 5)
ax.axis('off')

y_start     = len(data_raw) + 3.8
col1_x      = 0.4
col2_x      = 3.6
lw_main     = 2.0
lw_sub      = 0.8

# ── Header ────────────────────────────────────────────────────────────────────
ax.axhline(y=y_start + 0.8, xmin=0.04, xmax=0.96, color='black', linewidth=lw_main)
ax.text(col1_x, y_start + 0.25, "Dimensión",
        weight='bold', fontsize=13, family='serif')
ax.text(col2_x, y_start + 0.25, "Variables  (nativas  +  engineered)",
        weight='bold', fontsize=13, family='serif')
ax.axhline(y=y_start, xmin=0.04, xmax=0.96, color='black', linewidth=lw_sub)

# ── Fila Target ───────────────────────────────────────────────────────────────
current_y = y_start - 0.75
rect = patches.Rectangle((0.35, current_y - 0.5), 9.3, 0.75,
                          linewidth=0, facecolor='#fff9c4', zorder=0)
ax.add_patch(rect)
ax.text(col1_x, current_y, target_row[0],
        weight='bold', fontsize=12, family='serif', va='top', zorder=1)
ax.text(col2_x, current_y, target_row[1],
        weight='bold', fontsize=12, family='serif', va='top', zorder=1)
current_y -= 1.1

# ── Filas de datos ────────────────────────────────────────────────────────────
for i, (cat, vars_text) in enumerate(data_raw):
    # Fondo alternado suave
    num_lines  = vars_text.count('\n') + 1
    row_height = 0.45 + num_lines * 0.32
    if i % 2 == 0:
        bg = patches.Rectangle((0.35, current_y - row_height + 0.35), 9.3, row_height,
                                linewidth=0, facecolor='#f5f5f5', zorder=0)
        ax.add_patch(bg)

    ax.text(col1_x, current_y, cat,
            weight='bold', fontsize=10, family='serif', va='top', zorder=1)
    ax.text(col2_x, current_y, vars_text,
            fontsize=10, family='serif', va='top', zorder=1)
    current_y -= row_height

# ── Línea de cierre ───────────────────────────────────────────────────────────
ax.axhline(y=current_y + 0.2, xmin=0.04, xmax=0.96, color='black', linewidth=lw_main)

# ── Nota al pie ───────────────────────────────────────────────────────────────
ax.text(0.4, current_y - 0.1,
        "Nota: todas las variables engineered fueron ajustadas únicamente sobre Train para evitar data leakage.",
        fontsize=8, family='serif', color='gray', va='top')

# ── Guardar ───────────────────────────────────────────────────────────────────
plt.savefig(path_final, dpi=300, bbox_inches='tight',
            facecolor='white', transparent=False)
plt.close(fig)
print(f"✅ Tabla guardada en: {path_final} ")


✅ Tabla guardada en: Figures/FeatEng\01_Variables_finales_modelo.png 


In [35]:
!pip freeze > requirements_02_feateng.txt